In [0]:
import os
import sys

import mlflow
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from modelos import pronostico as P

DESDE = "2022-01-02"
SUBIDAS = [0.0, 1.0, 1.25, 1.5]   # 0 es la regla del plan; con 1,25 la roja pide que el pronóstico suba un 25 %
PRESUPUESTO = 1.0   # falsas alarmas por provincia y año que se aceptan al elegir la regla (por inicios de brote)
FECHAS_EXACTAS = "2024-03-03"   # SE 10 de 2024: desde aquí se sabe el día en que subió cada gaceta que usa un boletín
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/vigia-dengue-modelo")

In [0]:
features = spark.table("vigia.gold.features_provincia_semana").toPandas()
canal = spark.table("vigia.gold.canal_endemico").toPandas()
semanas = (spark.table("vigia.gold.dim_calendario_epi")
           .select("anio_epi", "semana_epi", "inicio_semana").distinct().toPandas())
f = P.preparar(features)
base = P.evaluacion(f, canal, semanas, DESDE)
serie = P.serie_final(f, canal, semanas)

columnas = ["modelo", "dpa_provin", "emision", "horizonte", "pronostico", "origen"]
guardados = [spark.table(t).select(columnas).toPandas()
             for t in ("vigia.ml.backtest_pronosticos", "vigia.ml.backtest_por_provincia")
             if spark.catalog.tableExists(t)]
pronosticos = P.unir(base, [P.de_spark(g) for g in guardados])
error = P.error_por_horizonte(pronosticos)
campeon = error.loc[error["modelo"].isin(["gbm_global", "gbm_sin_clima"]), "modelo"].iloc[0]
print("Modelo campeón (el global con menos error medio):", campeon)

In [0]:
# Las mismas reglas para el campeón y para las dos líneas base sin modelo: el promedio de las últimas 4 semanas
# (ingenuo) y la última semana conocida (ultimo). El campeón solo se justifica si avisa mejor que "ultimo", que
# está en el backtest desde que backtest.ipynb lo guarda
brotes = P.inicios_de_brote(serie)
modelos = [m for m in ("ingenuo", "ultimo", campeon) if m in set(pronosticos["modelo"])]
reglas, boletines, anticipaciones, avisos = [], [], [], []
for modelo in modelos:
    for subida in SUBIDAS:
        b = P.boletines(pronosticos, modelo, subida)
        reglas.append(P.resumen_regla(b, base, serie))
        boletines.append(b)
        anticipaciones.append(P.anticipacion(b, serie))
        avisos.append(P.avisos_de_inicio(b, brotes))
reglas = pd.DataFrame(reglas)
elegida = P.elegir_regla(reglas[reglas["modelo"] == campeon], PRESUPUESTO)
reglas["elegida"] = reglas.index == elegida
print(f"{len(brotes)} inicios de brote en {brotes['dpa_provin'].nunique()} provincias")
display(reglas.round(2))

# De casi todas las gacetas anteriores solo se sabe el mes de subida. Las mismas métricas de inicio, solo con los
# boletines desde FECHAS_EXACTAS: ahí el resultado no depende de qué extremo de la fecha de publicación se use
recientes = pd.DataFrame([dict(modelo=b["modelo"].iloc[0], subida=b["subida"].iloc[0],
                               **P.metricas_inicio(b[b["emision"] >= FECHAS_EXACTAS], serie)) for b in boletines])
display(recientes.round(2))

In [0]:
resumen = P.tabla_resumen(error, reglas)
tablas = {"backtest_resumen": resumen, "backtest_alertas": pd.concat(boletines, ignore_index=True),
          "backtest_anticipacion": pd.concat(anticipaciones, ignore_index=True),
          "backtest_inicios": pd.concat(avisos, ignore_index=True), "backtest_inicios_recientes": recientes}
for nombre, tabla in tablas.items():
    (spark.createDataFrame(P.para_spark(tabla)).write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"vigia.ml.{nombre}"))
display(resumen.round(2))

In [0]:
fila = reglas.loc[elegida]
regla = "regla del canal" if fila["subida"] == 0 else f"canal y subida ×{fila['subida']:g}".replace(".", ",")
a = anticipaciones[elegida]
figura = P.grafico_costa(serie, boletines[elegida], a, titulo=f"Alerta de 2024 en la Costa · {campeon}, {regla}")
display(figura)

m = a[a["dpa_provin"] == P.MANABI].iloc[0]
if m["avisada"]:
    print(f"Con lo que se sabía cada semana, Vigía Dengue habría alertado a Manabí {int(m['anticipacion'])} "
          f"semanas antes del pico de 2024 ({m['pico_se']}): la roja salió con la emisión de la "
          f"{m['primera_roja_se']}, publicada el {m['publicada']:%d/%m/%Y}.")
elif m["saturada"]:
    print(f"Manabí ya estaba en roja desde la {m['roja_desde_se']}: esta regla no distingue la subida de 2024.")
else:
    print("Esta regla no encendió la roja en Manabí antes del pico de 2024.")

reciente = recientes[(recientes["modelo"] == campeon) & (recientes["subida"] == fila["subida"])].iloc[0]
print(f"Inicios de brote avisados a tiempo con la regla elegida: {int(fila['inicios_a_tiempo'])} de "
      f"{int(fila['inicios'])} en todo el backtest y {int(reciente['inicios_a_tiempo'])} de "
      f"{int(reciente['inicios'])} con los boletines desde {FECHAS_EXACTAS}.")

with mlflow.start_run(run_name="alertas"):
    mlflow.log_params({"modelo": campeon, "subida": fila["subida"], "presupuesto": PRESUPUESTO, "piso": P.PISO})
    mlflow.log_metrics({k: float(fila[k]) for k in ("sensibilidad", "falsas_por_anio", "anticipacion_costa",
                                                    "anticipacion_manabi", "inicios", "inicios_a_tiempo",
                                                    "anticipacion_inicio", "falsas_inicio_por_anio")
                        if pd.notna(fila[k])})
    mlflow.log_figure(figura, "alerta_2024_costa.png")
    mlflow.log_text(resumen.to_csv(index=False), "backtest_resumen.csv")